# Basic setup to get all playlists of the YGO Card EU channel

In [283]:
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# 1. SETUP CONFIGURATION
# Make sure to replace this with your actual key from the Google Cloud Console
API_KEY = os.getenv("YOUTUBE_API_KEY")
HANDLE = "@YuGiOhCardEU"

# 2. DEFINE LOGIC FUNCTIONS
def get_channel_id_via_api(youtube, handle):
    """Uses the official YouTube API search method to securely resolve a handle 
    into an absolute Channel ID (UC...).
    """
    try:
        response = youtube.search().list(
            q=handle,
            type="channel",
            part="id,snippet",
            maxResults=1
        ).execute()
        
        items = response.get("items", [])
        if not items:
            print(f"Error: Could not find channel with handle: {handle}")
            return None
            
        channel_id = items[0]["id"]["channelId"]
        channel_title = items[0]["snippet"]["title"]
        print(f"Successfully connected to: '{channel_title}'")
        return channel_id
        
    except HttpError as e:
        print(f"An API error occurred while looking up the handle: {e}")
        return None

def fetch_all_playlists(youtube, channel_id):
    """Iterates through all pages of playlists using the YouTube Data API."""
    playlists = []
    next_page_token = None
    
    while True:
        try:
            response = youtube.playlists().list(
                channelId=channel_id,
                part="snippet,contentDetails",
                maxResults=50,
                pageToken=next_page_token
            ).execute()
            
            for item in response.get("items", []):
                playlists.append({
                    "Playlist Title": item["snippet"]["title"],
                    "Video Count": item["contentDetails"]["itemCount"],
                    "Playlist ID": item["id"]
                })
            next_page_token = response.get("nextPageToken")
            if not next_page_token:
                break
        except HttpError as e:
            print(f"An API error occurred while downloading lists: {e}")
            break
    return playlists

# Fetch all playlists into a dataframe

In [284]:
youtube = build("youtube", "v3", developerKey=os.getenv("YOUTUBE_API_KEY"))

channel_id = get_channel_id_via_api(youtube, HANDLE)

if channel_id:
    print(f"Channel ID confirmed as {channel_id}. Fetching playlist data...")
    data = fetch_all_playlists(youtube, channel_id)
    df = pd.DataFrame(data)
    print(f"Done! Extracted {len(df)} total playlists.\n")
    
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_colwidth', None)
    
    # Renders the table visually beneath the cell
    display(df) 
else:
    print("Execution halted: Unable to determine Channel ID.")

Successfully connected to: 'YuGiOhCardEU'
Channel ID confirmed as UCvjX8v8KMcLToVxKLjDNSVg. Fetching playlist data...
Done! Extracted 132 total playlists.



,Playlist Title,Video Count,Playlist ID
0,Yu-Gi-Oh! TCG | Battles of Legend: Glorious Gallery Unboxing Series,22,PLIm0Rt8gzRbskxI-lI7pC0klGQw5FLGBI
1,Yu-Gi-Oh! TCG | Blazing Dominion Unboxing Series,22,PLIm0Rt8gzRbtYPRjkHV_z8WgY1I42EJdc
2,Yu-Gi-Oh! Rarity Collection,4,PLIm0Rt8gzRbv2hsmR9PrFpmOlwSKVDArh
3,Yu-Gi-Oh! TCG Rarity Collection 5 Reveals!,21,PLIm0Rt8gzRbuQw2N4O0oOvc5gOnPUO8dz
4,Yu-Gi-Oh! MASTER DUEL EU Challenger Cup Final Paris 2026,2,PLIm0Rt8gzRbuomctdTErxkiYT6zPT07GZ
5,YCS Dortmund 2026,41,PLIm0Rt8gzRbs70xRsspnrAuR1qgUDXl4e
6,Yu-Gi-Oh! TCG Burst Protocol Reveals!,19,PLIm0Rt8gzRbu8mVV-2QrsgpJr-uX0Zgg6
7,Yu-Gi-Oh! DUEL LINKS,16,PLIm0Rt8gzRbt6U_yEBFCIXcPTWOfopKkK
8,THE CHRONICLES DECK: Spirit Charmers (All-Foil Edition),16,PLIm0Rt8gzRbu84BuZvx-VrDEZEJV6ODcU
9,Yu-Gi-Oh! CARD GAME THE CHRONICLES,12,PLIm0Rt8gzRbunisv8ir7jZl-KCmf6MHQE


# Filter tournaments by phrases

In [83]:
TARGET_PHRASES = ["YCS", "Championship", "WCQ", "National", "Open"]
strict_regex = "|".join([rf"\b{word}\b" for word in TARGET_PHRASES])
filtered_df = df[df["Playlist Title"].str.contains(strict_regex, case=False, na=False)]

print(f"--- Found {len(filtered_df)} playlists matching your target tournament phrases ---")
filtered_df = filtered_df.sort_values(by="Video Count", ascending=False)
display(filtered_df)

--- Found 71 playlists matching your target tournament phrases ---


,Playlist Title,Video Count,Playlist ID
96,YCS Bochum 2018 | Yu-Gi-Oh! Championship Series Bochum 2018,78,PLIm0Rt8gzRbv3fIxWySnaolII9TXeBGDl
130,WCQ: EC2015 | 2015 Yu-Gi-Oh! TCG WCQ: European Championship,60,PLIm0Rt8gzRbss2nX1mixsDYj4CpmZ8ugW
116,WCQ: EC2016 | 2016 WCQ: European Championship,57,PLIm0Rt8gzRbvYYuWPcfcb1ZNQLCoPT5vm
106,2017 WCQ: EC | 2017 WCQ: European Championship,54,PLIm0Rt8gzRbsHq8NXVNrIejQoUN3nMFfo
86,YCS Milan 2018 | Yu-Gi-Oh! Championship Series Milan 2018,50,PLIm0Rt8gzRbvNmWVsRbgwdxXWcX1EXxh0
13,YCS Bologna 2025,46,PLIm0Rt8gzRbuI3ij0C4yrfAQb8tMbtCt8
20,"Yu-Gi-Oh! World Championship 2025 - Paris, France",44,PLIm0Rt8gzRbvUqBmSmnIEr9nTIFuqAW_F
5,YCS Dortmund 2026,41,PLIm0Rt8gzRbs70xRsspnrAuR1qgUDXl4e
84,YCS Düsseldorf 2019 | Yu-Gi-Oh! Championship Series Düsseldorf 2019,40,PLIm0Rt8gzRbue3Nu1Xy67JzYHUGRFFCSR
94,2018 WCQ: European Championship,39,PLIm0Rt8gzRbuLKQ294o73x9SXXT4S-8IO


# Feature Match Processing

In [197]:
import re
import pandas as pd

# Global Regex Compilation Matrix
RE_EURO = re.compile(r"(?:WCQ:\s*)?\bEuropean\b(?:\s+Yu-Gi-Oh!)?\s+\bChampionship\b", re.IGNORECASE)
RE_NATS = re.compile(r"WCQ:\s*([A-Za-z]+)\s+National Championship", re.IGNORECASE)
RE_OPEN = re.compile(r"\b([A-Za-z]+)\s+Open\b", re.IGNORECASE)
RE_YEARS = re.compile(r'\b(19\d{2}|20\d{2})\b')


def clean_base_text(full_title):
    """Handles pipe splitting, string type conversions, and leading/trailing whitespace."""
    base_text = str(full_title).strip()
    if "|" in base_text:
        base_text = base_text.split("|")[0].strip()
    return base_text


def extract_tournament_year(text):
    """Finds the main tournament year and strips it out of the title string."""
    year_match = RE_YEARS.search(text)
    if year_match:
        year = year_match.group(1)
        clean_title = text.replace(year, "").replace("  ", " ").strip(" -:")
        return year, clean_title
    return "Unknown", text


def process_ycs_event(clean_title, base_text):
    """Applies specialized transformations for YCS, Remote Duels, and Time Wizard formats."""
    category = 'Advanced'
    is_remote = 'Remote Duel' in clean_title
    
    if any(kw in clean_title for kw in ['Time Wizard', 'Ultimate Time Wizard']) or any(fmt in base_text for fmt in ['Edison', 'Goat', 'Toss', 'HAT']):
        category = 'Time Wizard'
        clean_title = clean_title.replace('Ultimate Time Wizard', '').replace('Time Wizard', '').replace('Format', '')
        clean_title = RE_YEARS.sub('', clean_title)  # Wipe retro format years (e.g., 2010)

    if 'Genesys' in clean_title:
        category = 'Genesys'
        
    clean_title = clean_title.replace('Yu-Gi-Oh! Championship Series', 'YCS')

    if 'YCS' in clean_title:
        clean_title = 'YCS' + clean_title.split('YCS')[1]
        clean_title = re.sub(r'\bYCS\s+in\s+', 'YCS ', clean_title, flags=re.IGNORECASE)
        
    if is_remote:
        clean_title = 'Remote ' + clean_title

    if any(dash in clean_title for dash in ['–', '-']) and 'Top' in clean_title:
        split_char = '–' if '–' in clean_title else '-'
        parts = clean_title.split(split_char)
        clean_title = f"{parts[0].strip()} ({parts[1].strip()})"
        
    clean_title = re.sub(r'\s+', ' ', clean_title).strip(" -")
    return clean_title, category


def identify_other_categories(clean_title):
    """Checks for World Championships, Nationals, Opens, and Continental WCQs."""
    category = 'Advanced'
    
    # 1. World Championships
    if 'World Championship' in clean_title or 'WCS' in clean_title:
        if 'Dragon Duel' in clean_title:
            category = 'Dragon Duel'
        return 'World Championship', category
    
    # 2. National Championships
    if match_nat := RE_NATS.search(clean_title):
        country = match_nat.group(1).strip()
        return f"{country} National Championship", category
        
    # 3. Open Tournaments
    if match_open := RE_OPEN.search(clean_title):
        country = match_open.group(1).strip()
        country = 'UK' if country.lower() == 'uk' else country.capitalize()
        return f"{country} Open", category
        
    # 4. Continental WCQs
    if 'WCQ: EC' in clean_title or 'World Championship Qualifier' in clean_title or RE_EURO.search(clean_title):
        return 'European WCQ', category
        
    return clean_title, category


def extract_title_and_year(full_title):
    if 'interview' in str(full_title).lower():
        return pd.Series([None, None, None])
        
    base_text = clean_base_text(full_title)
    year, clean_title = extract_tournament_year(base_text)
    if 'Yu-Gi-Oh! Championship Series' in clean_title or 'YCS' in clean_title:
        clean_title, category = process_ycs_event(clean_title, base_text)
    else:
        clean_title, category = identify_other_categories(clean_title)
        
    return pd.Series([clean_title, year, category])

In [199]:
filtered_df[["Clean Title", "Year", "Category"]] = filtered_df["Playlist Title"].apply(extract_title_and_year)
cleaned_df = filtered_df[["Clean Title", "Year", "Category", "Video Count", "Playlist ID", "Playlist Title"]]
cleaned_df = cleaned_df.dropna(subset=["Clean Title"])

# 3. Display your perfect table
display(cleaned_df)

,Clean Title,Year,Category,Video Count,Playlist ID,Playlist Title
96,YCS Bochum,2018,Advanced,78,PLIm0Rt8gzRbv3fIxWySnaolII9TXeBGDl,YCS Bochum 2018 | Yu-Gi-Oh! Championship Series Bochum 2018
130,European WCQ,Unknown,Advanced,60,PLIm0Rt8gzRbss2nX1mixsDYj4CpmZ8ugW,WCQ: EC2015 | 2015 Yu-Gi-Oh! TCG WCQ: European Championship
116,European WCQ,Unknown,Advanced,57,PLIm0Rt8gzRbvYYuWPcfcb1ZNQLCoPT5vm,WCQ: EC2016 | 2016 WCQ: European Championship
106,European WCQ,2017,Advanced,54,PLIm0Rt8gzRbsHq8NXVNrIejQoUN3nMFfo,2017 WCQ: EC | 2017 WCQ: European Championship
86,YCS Milan,2018,Advanced,50,PLIm0Rt8gzRbvNmWVsRbgwdxXWcX1EXxh0,YCS Milan 2018 | Yu-Gi-Oh! Championship Series Milan 2018
13,YCS Bologna,2025,Advanced,46,PLIm0Rt8gzRbuI3ij0C4yrfAQb8tMbtCt8,YCS Bologna 2025
20,World Championship,2025,Advanced,44,PLIm0Rt8gzRbvUqBmSmnIEr9nTIFuqAW_F,"Yu-Gi-Oh! World Championship 2025 - Paris, France"
5,YCS Dortmund,2026,Advanced,41,PLIm0Rt8gzRbs70xRsspnrAuR1qgUDXl4e,YCS Dortmund 2026
84,YCS Düsseldorf,2019,Advanced,40,PLIm0Rt8gzRbue3Nu1Xy67JzYHUGRFFCSR,YCS Düsseldorf 2019 | Yu-Gi-Oh! Championship Series Düsseldorf 2019
94,European WCQ,2018,Advanced,39,PLIm0Rt8gzRbuLKQ294o73x9SXXT4S-8IO,2018 WCQ: European Championship


# Get playlist IDs

In [185]:
playlist_ids = cleaned_df['Playlist ID'].astype(str).tolist()
playlist_ids

['PLIm0Rt8gzRbv3fIxWySnaolII9TXeBGDl',
 'PLIm0Rt8gzRbss2nX1mixsDYj4CpmZ8ugW',
 'PLIm0Rt8gzRbvYYuWPcfcb1ZNQLCoPT5vm',
 'PLIm0Rt8gzRbsHq8NXVNrIejQoUN3nMFfo',
 'PLIm0Rt8gzRbvNmWVsRbgwdxXWcX1EXxh0',
 'PLIm0Rt8gzRbuI3ij0C4yrfAQb8tMbtCt8',
 'PLIm0Rt8gzRbvUqBmSmnIEr9nTIFuqAW_F',
 'PLIm0Rt8gzRbs70xRsspnrAuR1qgUDXl4e',
 'PLIm0Rt8gzRbue3Nu1Xy67JzYHUGRFFCSR',
 'PLIm0Rt8gzRbuLKQ294o73x9SXXT4S-8IO',
 'PLIm0Rt8gzRbvpM61P9xfJEZoU558nP7Hw',
 'PLIm0Rt8gzRbudSiIYYy2neWhQqecruQNU',
 'PLIm0Rt8gzRbtnbZYH99PGTqCWtR6WZS77',
 'PLIm0Rt8gzRbuSa9zamQ9XAqfN9f5wa0IM',
 'PLIm0Rt8gzRbutxiGy8fJI0W3m4bcGg3-j',
 'PLIm0Rt8gzRbsro5ChBPW-Jv8q2Potz2ll',
 'PLIm0Rt8gzRbus2XHbGg1A0qef58bLUg9W',
 'PLIm0Rt8gzRbvQIZ4LdDzsQcg05qD5-AfI',
 'PLIm0Rt8gzRbvARnj3qZUwXVaN3WLiZu-G',
 'PLIm0Rt8gzRbsCXQnGZVhu2Acnfbi43uk6',
 'PLIm0Rt8gzRbtPVaqIwILHqX2YvVk54upH',
 'PLIm0Rt8gzRbtzapH_8KLPAehZVrHsALMR',
 'PLIm0Rt8gzRbu5rDW3wVN5G_Zcg_G53RB9',
 'PLIm0Rt8gzRbsuyvjSAe5Zf9_CxIvEqsH2',
 'PLIm0Rt8gzRbtzFNzn1ayewRiNuwNekWsq',
 'PLIm0Rt8gzRbvdbAcXuDGBP

In [305]:
from googleapiclient.discovery import build
import pandas as pd

def get_playlists_videos(playlist_ids):
    """
    Iterates through a list of YouTube playlist IDs, fetches all video titles 
    and descriptions, and returns them in a pandas DataFrame.
    """
    # Initialize the YouTube API client
    # Replace 'YOUR_API_KEY' with your actual YouTube Data API v3 key
    youtube = build('youtube', 'v3', developerKey=os.getenv("YOUTUBE_API_KEY"))
    
    video_data = []
    
    for playlist_id in playlist_ids:
        next_page_token = None
        
        # Loop to handle pagination (playlists with more than 50 items)
        while True:
            # Request playlist items
            request = youtube.playlistItems().list(
                part="snippet",
                playlistId=playlist_id,
                maxResults=50,
                pageToken=next_page_token
            )
            response = request.execute()
            
            # Extract title and description from each video
            for item in response.get('items', []):
                title = item['snippet']['title']
                description = item['snippet']['description']
                
                video_data.append({
                    'Playlist ID': playlist_id,
                    'Title': title,
                    'Description': description
                })
            
            # Check if there's another page of videos
            next_page_token = response.get('nextPageToken')
            if not next_page_token:
                break
                
    # Convert the list of dictionaries into a Pandas DataFrame
    df = pd.DataFrame(video_data)
    return df

playlists = get_playlists_videos(playlist_ids)

In [306]:
playlists

,Playlist ID,Title,Description
0,PLIm0Rt8gzRbv3fIxWySnaolII9TXeBGDl,YCS Bochum 2018: Welcome,"Footage recorded from Yu-Gi-Oh! YCS Bochum, held on February 24/25, 2018.\nWelcome to YCS\n\nYu-Gi-Oh! TRADING CARD GAME\n© 1996 Kazuki Takahashi\n© 2014 NAS • TV TOKYO\n\n\nWeb: http://www.yugioh-card.com\n\nFacebook: http://facebook.com/YuGiOhTCGEU\nTwitch: http://twitch.tv/OfficialYuGiOhChannel\n\nLive Event Coverage: www.yugioh-card.com/uk/live/\nCoverage Blog: http://konami-europe.com/coverage"
1,PLIm0Rt8gzRbv3fIxWySnaolII9TXeBGDl,YCS Bochum 2018: Event Spotlight,"Footage recorded from Yu-Gi-Oh! YCS Bochum, held on February 24/25, 2018.\nEvent Spotlight\n\nYu-Gi-Oh! TRADING CARD GAME\n© 1996 Kazuki Takahashi\n© 2014 NAS • TV TOKYO\n\n\nWeb: http://www.yugioh-card.com\n\nFacebook: http://facebook.com/YuGiOhTCGEU\nTwitch: http://twitch.tv/OfficialYuGiOhChannel\n\nLive Event Coverage: www.yugioh-card.com/uk/live/\nCoverage Blog: http://konami-europe.com/coverage"
2,PLIm0Rt8gzRbv3fIxWySnaolII9TXeBGDl,YCS Bochum 2018: Quick Questions,"Footage recorded from Yu-Gi-Oh! YCS Bochum, held on February 24/25, 2018.\nQuick Questions\n\nYu-Gi-Oh! TRADING CARD GAME\n© 1996 Kazuki Takahashi\n© 2014 NAS • TV TOKYO\n\n\nWeb: http://www.yugioh-card.com\n\nFacebook: http://facebook.com/YuGiOhTCGEU\nTwitch: http://twitch.tv/OfficialYuGiOhChannel\n\nLive Event Coverage: www.yugioh-card.com/uk/live/\nCoverage Blog: http://konami-europe.com/coverage"
3,PLIm0Rt8gzRbv3fIxWySnaolII9TXeBGDl,YCS Bochum 2018: Meta Discussion,"Footage recorded from Yu-Gi-Oh! YCS Bochum, held on February 24/25, 2018.\nMeta Discussion\n\nYu-Gi-Oh! TRADING CARD GAME\n© 1996 Kazuki Takahashi\n© 2014 NAS • TV TOKYO\n\n\nWeb: http://www.yugioh-card.com\n\nFacebook: http://facebook.com/YuGiOhTCGEU\nTwitch: http://twitch.tv/OfficialYuGiOhChannel\n\nLive Event Coverage: www.yugioh-card.com/uk/live/\nCoverage Blog: http://konami-europe.com/coverage"
4,PLIm0Rt8gzRbv3fIxWySnaolII9TXeBGDl,YCS Bochum 2018: Giant Card Spotlight,"Footage recorded from Yu-Gi-Oh! YCS Bochum, held on February 24/25, 2018.\nGiant Card Spotlight\n\nYu-Gi-Oh! TRADING CARD GAME\n© 1996 Kazuki Takahashi\n© 2014 NAS • TV TOKYO\n\n\nWeb: http://www.yugioh-card.com\n\nFacebook: http://facebook.com/YuGiOhTCGEU\nTwitch: http://twitch.tv/OfficialYuGiOhChannel\n\nLive Event Coverage: www.yugioh-card.com/uk/live/\nCoverage Blog: http://konami-europe.com/coverage"
5,PLIm0Rt8gzRbv3fIxWySnaolII9TXeBGDl,YCS Bochum 2018: Prizes Spotlight,"Footage recorded from Yu-Gi-Oh! YCS Bochum, held on February 24/25, 2018.\nPrizes Spotlight\n\nYu-Gi-Oh! TRADING CARD GAME\n© 1996 Kazuki Takahashi\n© 2014 NAS • TV TOKYO\n\n\nWeb: http://www.yugioh-card.com\n\nFacebook: http://facebook.com/YuGiOhTCGEU\nTwitch: http://twitch.tv/OfficialYuGiOhChannel\n\nLive Event Coverage: www.yugioh-card.com/uk/live/\nCoverage Blog: http://konami-europe.com/coverage"
6,PLIm0Rt8gzRbv3fIxWySnaolII9TXeBGDl,YCS Bochum 2018: Round 1 Feature Match,"Footage recorded from Yu-Gi-Oh! YCS Bochum, held on February 24/25, 2018.\nRound 1: Jörg Müller vs. Eugen Heidt\n \n Yu-Gi-Oh! TRADING CARD GAME\n © 1996 Kazuki Takahashi\n © 2014 NAS • TV TOKYO\n \n Web: www.yugioh-card.com\n \n Facebook: http://facebook.com/YuGiOhTCGEU\n Twitch: http://twitch.tv/OfficialYuGiOhChannel\n \n Live Event Coverage: www.yugioh-card.com/uk/live/\n Event Blog: http://coverage.yugioh-card.com -- Watch live at https://www.twitch.tv/officialyugiohc... -- Watch live at https://www.twitch.tv/officialyugiohc... -- Watch live at https://www.twitch.tv/officialyugiohchannel"
7,PLIm0Rt8gzRbv3fIxWySnaolII9TXeBGDl,YCS Bochum 2018: Round 1 Feature Match Interview,"Footage recorded from Yu-Gi-Oh! YCS Bochum, held on February 24/25, 2018.\nInterview with the winner of Round 1 Feature Match\n \n Yu-Gi-Oh! TRADING CARD GAME\n © 1996 Kazuki Takahashi\n © 2014 NAS • TV TOKYO\n \n Web: www.yugioh-card.com\n \n Facebook: http://facebook.com/YuGiOhTCGEU\n Twitch: http://twitch.tv/OfficialYuGiOhChannel\n \n Live Event C

# Getting an LLM to do the tagging is easier than doing continuous regex changes with how Konami inconsistently tags their videos.

In [223]:
import json
import time
from pydantic import BaseModel
# Using the updated Google GenAI SDK imports
from google import genai
from google.genai import types

# Initialize your client (it automatically picks up GEMINI_API_KEY from environment)
client = genai.Client()

# 1. Update your Schema structure
class MatchRecord(BaseModel):
    is_match: bool
    round_name: str
    player_1: str
    deck_1: str
    player_2: str
    deck_2: str

# 2. Define the parser function with explicit logic for edge cases
def gemini_parse_match(title, description):
    prompt = f"""
    You are an expert tournament data extractor for the Yu-Gi-Oh! TCG. 
    Analyze the video title and description to extract match data.
    
    ### CRITICAL FILTERING RULES:
    1. A video is a valid match (`is_match`: true) ONLY if it is an official, standard multi-player tournament match featuring two active players and an explicitly stated round or top-cut designation (e.g., Round 1, Top 8, Finals).
    2. Set `is_match`: false if the video is:
       - A single-player focus or a single-deck highlight video that doesn't name both active competitors (e.g., "Mirror Match - Jesse Kotton").
       - An interview or a "Post Match Discussion and Interview" (talking *after* the match concluded).
       - An exhibition match, fun match, staff duel, or "Caster vs Master" challenge.
       - A "Deck Profile", "Spotlight", "Ceremony", "Vlog", "Highlights", or casual discussion.
       
    ### DATA EXTRACTION RULES (Only if `is_match` is true):
    - If a field is missing, use "Unknown". Do not guess.
    - Remove all flag emojis, country bracket prefixes (e.g., [DE], [UK]), and trailing tournament metadata from names.
    - DECK VS DECK MATCHES: If the title shows two archetypes fighting directly without player names (e.g., "Lunalight vs Psychic" or "Stardust Dragon vs Frogs"), map the archetypes to `deck_1` and `deck_2`, and set `player_1` and `player_2` to "Unknown".

    TITLE: {title}
    DESCRIPTION: {description}
    """
    
    # Using the correct SDK method for Structured JSON Outputs
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=MatchRecord,
            temperature=0.1
        ),
    )
    return json.loads(response.text)

# 3. Execution loop
processed_dataset = []

# Mock array assumed: replace `data` with your actual input data structure
for entry in data:
    try:
        title_snippet = entry.get('Video Title', '')[:40]
        print(f"Analyzing: {title_snippet}...")
        
        result = gemini_parse_match(entry.get('Video Title', ''), entry.get('Video Description', ''))
        
        # We only keep the clean records that are actual match gameplay
        if result.get("is_match"):
            # Add metadata for audit tracking
            result['original_title'] = entry.get('Video Title')
            processed_matches.append(result)
            print(f"   ✅ SUCCESS: Match found! {result['player_1']} vs {result['player_2']}")
        else:
            print("   ⏩ SKIPPED: Non-match content.")
            
        # --- THE SAFETY THROTTLE ---
        # 15 seconds keeps you well under the free-tier rate limit threshold
        time.sleep(15) 
            
    except Exception as e:
        print(f"❌ Error processing {title_snippet}: {e}")
        if "429" in str(e):
            print("⚠️ Resource exhausted (Rate limit). Pausing for 60 seconds...")
            time.sleep(60)

# 4. View results
print(f"\nProcessing complete. Successfully extracted {len(processed_matches)} live matches.")

Analyzing: YCS Bochum 2018: Welcome...
Analyzing: YCS Bochum 2018: Event Spotlight...
Analyzing: YCS Bochum 2018: Quick Questions...
Analyzing: YCS Bochum 2018: Meta Discussion...
Analyzing: YCS Bochum 2018: Giant Card Spotlight...
Analyzing: YCS Bochum 2018: Prizes Spotlight...
Analyzing: YCS Bochum 2018: Round 1 Feature Match...


KeyboardInterrupt: 